# ML Lab Practical 3: Customer Churn Prediction Using ANN
**Algorithm:** Artificial Neural Network (ANN) with Dropout and Early Stopping

**Dataset:** Churn_Modelling.csv

**Objective:** Predict whether a bank customer will exit (churn).


In [ ]:
# 1. Import Libraries
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, confusion_matrix

from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout, Input
from tensorflow.keras.callbacks import EarlyStopping


In [ ]:
# 2. Read the Dataset
data = pd.read_csv("Churn_Modelling.csv")

print("Dataset Shape:", data.shape)
print(data.head())

In [ ]:
# 3. Seperate Features and Target
data = data.drop(["RowNumber", "CustomerId", "Surname"], axis=1)

data = pd.get_dummies(
    data,
    columns=["Geography", "Gender"],
    drop_first=True
)

X = data.drop("Exited", axis=1)
y = data["Exited"]

print("Feature shape:", X.shape)
print("Target shape:", y.shape)

In [ ]:
# 4. Split Data into Training and Testing Sets
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

In [ ]:
# 5. Normalize the Data
scaler = StandardScaler()

X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

In [ ]:
# 6. Build the initial Neural Network
initial_model = Sequential([
    Input(shape=(X_train.shape[1],)),
    Dense(64, activation="relu"),
    Dense(32, activation="relu"),
    Dense(1, activation="sigmoid")
])

initial_model.compile(
    optimizer="adam",
    loss="binary_crossentropy",
    metrics=["accuracy"]
)

initial_model.summary()

In [ ]:
# 7. Improve the Model using Dropping and Early Stopping
model = Sequential([
    Input(shape=(X_train.shape[1],)),
    Dense(64, activation="relu"),
    Dropout(0.3),
    Dense(32, activation="relu"),
    Dropout(0.2),
    Dense(16, activation="relu"),
    Dense(1, activation="sigmoid")
])

model.compile(
    optimizer="adam",
    loss="binary_crossentropy",
    metrics=["accuracy"]
)

early_stop = EarlyStopping(
    monitor="val_loss",
    patience=5,
    restore_best_weights=True
)

model.summary()

In [ ]:
# 8. Train the Improved Model
history = model.fit(
    X_train,
    y_train,
    epochs=50,
    batch_size=32,
    validation_split=0.2,
    callbacks=[early_stop],
    verbose=1
)

In [ ]:
# 9. Predict the Test Data
y_pred_prob = model.predict(X_test)
y_pred = (y_pred_prob > 0.5).astype(int).ravel()

In [ ]:
# 10. Accuracy Score
accuracy = accuracy_score(y_test, y_pred)
print("\nAccuracy Score:", accuracy)

In [ ]:
# 11. Confusion Matrix
cm = confusion_matrix(y_test, y_pred)
print("\nConfusion Matrix:")
print(cm)